# เริ่มต้นใช้ Webull OpenAPI: แบบฝึกหัดออฟไลน์

ตัวอย่างทั้ง 5 ใช้ข้อมูลสมมติและ Python standard library เท่านั้น ไม่เรียกเครือข่าย ไม่รับ credentials และไม่ส่งคำสั่งซื้อขาย

อ่านบทเรียน: https://nutdnuy.github.io/robo-trade-notes/chapter-11.html

Host และชื่อสถานะอ้างอิงเอกสาร Webull Thailand ตรวจเมื่อ 23 กันยายน 2026: https://developer.webull.co.th/apis/docs/sdk/ และ https://developer.webull.co.th/apis/docs/authentication/token/

เลือก Kernel Python 3 แล้วใช้ Run All ผลรันที่บันทึกไว้ใช้ Python 3.9; โค้ดนี้ไม่ต้องติดตั้ง Webull SDK


## 1. Choose an environment explicitly

เลือก test หรือ production ให้ชัดเจน ทดลองสะกดผิดแล้วดูว่าฟังก์ชันปฏิเสธ แทนที่จะเลือก production ให้อัตโนมัติ

In [1]:
HOSTS = {
    "test": "th-api.uat.webullbroker.com",
    "production": "api.webull.co.th",
}

def choose_host(environment):
    if environment not in HOSTS:
        raise ValueError("Choose test or production explicitly")
    return HOSTS[environment]

assert choose_host("test") == "th-api.uat.webullbroker.com"
try:
    choose_host("prodution")  # A typo must never silently choose production.
except ValueError:
    print("Unknown environment rejected")
else:
    raise AssertionError("Unknown environment was accepted")
print("Selected host:", choose_host("test"))

Unknown environment rejected
Selected host: th-api.uat.webullbroker.com


## 2. Interpret token status (a teaching example, not an SDK implementation)

เปลี่ยน status เป็น NORMAL แล้วเปรียบเทียบคำอธิบาย นี่เป็นตารางอธิบายสถานะ ไม่ใช่ระบบสร้าง Token จริง

In [2]:
TOKEN_ACTIONS = {
    "PENDING": "Complete verification in the Webull app",
    "NORMAL": "Token is active; check endpoint permissions separately",
    "EXPIRED": "Verification window elapsed; check the SDK token flow",
    "INVALID": "Token is unusable; check the SDK token flow",
}
status = "PENDING"
assert status != "NORMAL"
print(status, "->", TOKEN_ACTIONS.get(status, "Consult the current API documentation"))

PENDING -> Complete verification in the Webull app


## 3. Convert numeric strings and calculate a spread

ราคาสมมติ DEMO: bid 99.98 และ ask 100.02 USD ต่อหุ้น แปลง string ด้วย Decimal แล้วตรวจ spread เท่ากับ 0.04

In [3]:
import json
from decimal import Decimal

# Synthetic prices in USD per share; these were not returned by Webull.
snapshot = json.loads('{"symbol": "DEMO", "bid": "99.98", "ask": "100.02"}')
bid, ask = Decimal(snapshot["bid"]), Decimal(snapshot["ask"])
assert bid > 0 and ask >= bid
spread = ask - bid
assert spread == Decimal("0.04")
print(f'{snapshot["symbol"]} spread: {spread:.2f} USD per share')

DEMO spread: 0.04 USD per share


## 4. Interpret an explicitly specified timestamp unit

ตัวอย่างกำหนดหน่วย milliseconds และ timezone UTC ไว้ชัดเจน ใช้นาฬิกาสมมติที่ช้ากว่าเวลา quote 30 วินาที จึงรันซ้ำได้คำตอบเดิม ข้อมูลจริงต้องอ่าน schema ของ endpoint

In [4]:
from datetime import datetime, timezone, timedelta

# Synthetic millisecond timestamp, chosen only for this exercise.
quote_time_ms = 1788485400000
epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
quote_time = epoch + timedelta(milliseconds=quote_time_ms)
example_now = quote_time + timedelta(seconds=30)
age_seconds = (example_now - quote_time).total_seconds()
assert age_seconds == 30.0
print("Quote time (UTC):", quote_time.isoformat())
print("Age relative to the example clock:", age_seconds, "seconds")
# In production, use the endpoint's documented unit and timezone, not a guess.

Quote time (UTC): 2026-09-04T01:30:00+00:00
Age relative to the example clock: 30.0 seconds


## 5. Keep evidence for each permission separate

HTTP 200 พร้อมบัญชีที่คาดไว้เป็นหลักฐานของคำขออ่านบัญชีเท่านั้น ลองเปลี่ยนผลตรวจแต่ละรายการและอธิบายว่าส่วนใดยังไม่ผ่าน ไม่ใช้ตารางนี้รับรองความพร้อมซื้อขายจริง

In [5]:
# All values here are simulated observations, not a live readiness test.
checks = {
    "environment": "test",
    "token_status": "NORMAL",
    "account_http_status": 200,
    "expected_account_present": True,
    "market_data_request_succeeded": False,
}
account_read_verified = (
    checks["token_status"] == "NORMAL"
    and checks["account_http_status"] == 200
    and checks["expected_account_present"]
)
market_data_verified = checks["market_data_request_succeeded"]
assert account_read_verified and not market_data_verified
print("Account read verified:", account_read_verified)
print("Market data verified:", market_data_verified)
print("Next: verify market-data access separately. No trade was submitted.")

Account read verified: True
Market data verified: False
Next: verify market-data access separately. No trade was submitted.
